In [1]:
!pip -q install h5py numpy scikit-learn matplotlib tensorflow
import os, h5py, numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
import tensorflow as tf
print("TF version:", tf.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 726.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


TF version: 2.20.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/ADC2021-examplecode-main"  # change if your folder name is different
!ls -lh "$DATA_DIR"

Mounted at /content/drive
total 3.8G
-rw------- 1 root root 1.1K Jan 27 09:56 00_check_files.py
-rw------- 1 root root 2.6K Jan 27 09:58 01_train_dense.py
-rw------- 1 root root 2.1K Jan 27 10:05 01_train_dense_sklearn.py
-rw------- 1 root root 4.2K Jan 27 11:34 02_eval_signal_and_blackbox.py
-rw------- 1 root root 5.2M Jan 29 09:27 Ato4l_lepFilter_13TeV.h5
-rw------- 1 root root 138M Jan 27 08:25 background_for_training.h5
-rw------- 1 root root 1.7G Jan 27 09:30 BKG_dataset.h5
-rw------- 1 root root 391K Jan 27 11:34 bkg_mse.npy
-rw------- 1 root root 152M Jan 25 19:59 BlackBox_13TeV_PU20.h5
-rw------- 1 root root  48M Jan 27 05:32 BlackBox_background_mix.h5
-rw------- 1 root root  11K Jan 27 07:47 computeFLOPs.ipynb
-rw------- 1 root root  11K Jan 27 07:47 Convolutional_AE.ipynb
-rw------- 1 root root 3.4K Jan 27 07:47 create_datasets.py
-rw------- 1 root root  11K Jan 27 07:47 Dense_AE.ipynb
-rw------- 1 root root  42K Jan 27 11:12 dense_ae_sklearn.joblib
-rw------- 1 root root 1.6

In [3]:
def inspect_h5(path):
    print("\nFILE:", os.path.basename(path))
    with h5py.File(path, "r") as f:
        print("Keys:", list(f.keys()))
        for k in f.keys():
            try:
                print(" ", k, f[k].shape, f[k].dtype)
            except:
                print(" ", k, "no shape")

inspect_h5(os.path.join(DATA_DIR, "train.h5"))
inspect_h5(os.path.join(DATA_DIR, "val.h5"))
inspect_h5(os.path.join(DATA_DIR, "test.h5"))
inspect_h5(os.path.join(DATA_DIR, "BlackBox_13TeV_PU20.h5"))



FILE: train.h5
Keys: ['table']
  table no shape

FILE: val.h5
Keys: ['table']
  table no shape

FILE: test.h5
Keys: ['table']
  table no shape

FILE: BlackBox_13TeV_PU20.h5
Keys: ['EvtId', 'Particles', 'Particles_Classes', 'Particles_Names']
  EvtId (4210492,) int64
  Particles (4210492, 19, 4) float64
  Particles_Classes (4,) |S16
  Particles_Names (4,) |S5


In [4]:
inspect_h5(os.path.join(DATA_DIR, "BKG_dataset.h5"))


FILE: BKG_dataset.h5
Keys: ['X_test', 'X_train', 'X_val']
  X_test (800000, 57) float64
  X_train (2560000, 57) float64
  X_val (640000, 57) float64


In [5]:
import numpy as np
import h5py, os

def load_bkg_splits(path):
    with h5py.File(path, "r") as f:
        X_train = np.array(f["X_train"])
        X_val   = np.array(f["X_val"])
        X_test  = np.array(f["X_test"])
    return X_train, X_val, X_test

X_train, X_val, X_test = load_bkg_splits(os.path.join(DATA_DIR, "BKG_dataset.h5"))
print(X_train.shape, X_val.shape, X_test.shape)


(2560000, 57) (640000, 57) (800000, 57)


In [6]:
def preprocess(X):
    X = X.astype("float32").copy()

    # Keep only first 3 features (pT, eta, phi)
    # This matches the common AE setup and gives 19*3 = 57 features
    if X.shape[-1] == 4:
        X = X[:, :, :3]

    # Optional: log scale pT (feature 0)
    X[..., 0] = np.log1p(np.maximum(X[..., 0], 0))

    # Flatten to (N, 57)
    return X.reshape(X.shape[0], -1)


X_train_f = preprocess(X_train)
X_val_f   = preprocess(X_val)
X_test_f  = preprocess(X_test)

mean = X_train_f.mean(axis=0, keepdims=True)
std  = X_train_f.std(axis=0, keepdims=True) + 1e-6

X_train_n = (X_train_f - mean) / std
X_val_n   = (X_val_f   - mean) / std
X_test_n  = (X_test_f  - mean) / std

print("Final input shape:", X_train_n.shape)


Final input shape: (2560000, 57)


In [7]:
from tensorflow import keras
from tensorflow.keras import layers

inp = keras.Input(shape=(X_train_n.shape[1],))

x = layers.Dense(128, activation="relu")(inp)
x = layers.Dense(64, activation="relu")(x)
z = layers.Dense(16, activation="relu")(x)

x = layers.Dense(64, activation="relu")(z)
x = layers.Dense(128, activation="relu")(x)
out = layers.Dense(X_train_n.shape[1], activation=None)(x)

ae = keras.Model(inp, out)
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
ae.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 57)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         7,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 57)             │         7,353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,481 (130.79 KB)

 Trainable params: 33,481 (130.79 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = ae.fit(
    X_train_n, X_train_n,
    validation_data=(X_val_n, X_val_n),
    epochs=15,
    batch_size=2048
)

Epoch 1/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.6631 - val_loss: 0.3774
Epoch 2/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.3009 - val_loss: 0.2671
Epoch 3/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.2323 - val_loss: 0.1768
Epoch 4/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.1554 - val_loss: 0.1979
Epoch 5/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.1535 - val_loss: 0.1462
Epoch 6/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.1173 - val_loss: 0.1366
Epoch 7/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.1282 - val_loss: 0.1678
Epoch 8/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.1185 - val_loss: 0.1705
Epoch 9/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.1035 - val_loss: 0.1305
Epoch 10/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.1179 - val_loss: 0.1215
Epoch 11/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.1130 - val_loss: 0.1189
Epoch 12/15
1250/1250 ━━━━━━━━

In [9]:
def anomaly_score(model, Xn, batch_size=4096):
    Xhat = model.predict(Xn, batch_size=batch_size, verbose=0)
    mse = np.mean((Xn - Xhat)**2, axis=1)
    return mse

bkg_scores = anomaly_score(ae, X_test_n)
print("Background scores:", bkg_scores.min(), bkg_scores.mean(), bkg_scores.max())


Background scores: 0.0013701244 0.079307035 3312.9263


In [10]:
import h5py
import numpy as np

def load_particles(path, max_events=None):
    with h5py.File(path, "r") as f:
        X = f["Particles"][:max_events] if max_events else f["Particles"][:]
    return X


In [11]:
print("mean shape:", mean.shape)
print("std shape:", std.shape)
print("bkg_scores shape:", bkg_scores.shape)

mean shape: (1, 57)
std shape: (1, 57)
bkg_scores shape: (800000,)


In [12]:
signal_files = [
    "Ato4l_lepFilter_13TeV.h5",
    "hToTauTau_13TeV_PU20.h5",
    "hChToTauNu_13TeV_PU20.h5",
    "leptoquark_LOWMASS_lepFilter_13TeV.h5",
]

for sf in signal_files:
    spath = os.path.join(DATA_DIR, sf)
    if not os.path.exists(spath):
        print("Missing:", sf)
        continue

    X_sig = preprocess(load_particles(spath))
    X_sig_n = (X_sig - mean) / std

    sig_scores = anomaly_score(ae, X_sig_n)

    y = np.concatenate([np.zeros_like(bkg_scores), np.ones_like(sig_scores)])
    s = np.concatenate([bkg_scores, sig_scores])

    auc = roc_auc_score(y, s)
    print(f"{sf:40s}  ROC-AUC = {auc:.4f}")


Ato4l_lepFilter_13TeV.h5                  ROC-AUC = 0.9827
hToTauTau_13TeV_PU20.h5                   ROC-AUC = 0.9424
hChToTauNu_13TeV_PU20.h5                  ROC-AUC = 0.9735
leptoquark_LOWMASS_lepFilter_13TeV.h5     ROC-AUC = 0.9703


In [14]:
import os
import h5py

DATA_DIR = "/content/drive/MyDrive/ADC2021-examplecode-main" # Re-define DATA_DIR
bb_path = os.path.join(DATA_DIR, "BlackBox_13TeV_PU20.h5")

with h5py.File(bb_path, "r") as f:
    X_bb = f["Particles"][:]
    evtid = f["EvtId"][:]

X_bb_f = preprocess(X_bb)
X_bb_n = (X_bb_f - mean) / std

bb_scores = anomaly_score(ae, X_bb_n)

topk = 1000
idx = np.argsort(-bb_scores)[:topk]  # descending
top_evtid = evtid[idx]
top_scores = bb_scores[idx]

out_path = "/content/adc2021_top1000.csv"
import pandas as pd
pd.DataFrame({"EvtId": top_evtid, "score": top_scores}).to_csv(out_path, index=False)

print("Saved:", out_path)
print("First 10 IDs:", top_evtid[:10])

Saved: /content/adc2021_top1000.csv
First 10 IDs: [2933923 1222540 3447383 1712357 3442864 2613060 2782469 3744875 3487455
 3217149]


In [15]:
from sklearn.metrics import roc_curve

def tpr_at_fpr(y_true, scores, target_fpr=1e-4):
    fpr, tpr, thr = roc_curve(y_true, scores)
    # find nearest fpr <= target
    idx = (fpr <= target_fpr).nonzero()[0]
    if len(idx) == 0:
        return 0.0
    return tpr[idx[-1]]

# Example for one signal
y = np.concatenate([np.zeros_like(bkg_scores), np.ones_like(sig_scores)])
s = np.concatenate([bkg_scores, sig_scores])

for f in [1e-3, 1e-4, 1e-5]:
    print("TPR@FPR", f, "=", tpr_at_fpr(y, s, f))


TPR@FPR 0.001 = 0.06896906126667919
TPR@FPR 0.0001 = 0.007211990227400864
TPR@FPR 1e-05 = 0.0011657818079308401


# Task
Calculate the 99.9th percentile of the `bkg_scores` to set the anomaly threshold.

## Calculate anomaly threshold

### Subtask:
Determine a threshold based on a high percentile (e.g., 99.9th percentile) of the background anomaly scores (bkg_scores). This threshold will differentiate between normal and anomalous data points.


**Reasoning**:
To determine the anomaly threshold, I will calculate the 99.9th percentile of the background anomaly scores using numpy.percentile, as requested in the instructions.



In [16]:
anomaly_threshold = np.percentile(bkg_scores, 99.9)
print(f"Anomaly Threshold (99.9th percentile of background scores): {anomaly_threshold:.4f}")

Anomaly Threshold (99.9th percentile of background scores): 8.4339


## Classify BlackBox data

### Subtask:
Apply the calculated threshold to the anomaly scores of the BlackBox dataset (`bb_scores`) to classify each entry as either 'normal' or 'anomaly'.


**Reasoning**:
I will create a boolean array to identify anomalies by comparing each BlackBox score to the previously calculated anomaly threshold. This directly addresses the first instruction of the subtask.



In [17]:
is_anomaly = bb_scores > anomaly_threshold
print(f"Number of anomalies detected: {np.sum(is_anomaly)}")
print(f"Percentage of anomalies detected: {np.sum(is_anomaly) / len(bb_scores) * 100:.2f}%")

Number of anomalies detected: 5767
Percentage of anomalies detected: 0.14%


**Reasoning**:
I will create a new array, `blackbox_classification`, using `np.where` to assign 'anomaly' or 'normal' strings based on the `is_anomaly` boolean array, thereby fulfilling the second instruction of the subtask.



In [18]:
blackbox_classification = np.where(is_anomaly, 'anomaly', 'normal')
print("First 10 classifications:", blackbox_classification[:10])
print("Count of 'anomaly':", np.sum(blackbox_classification == 'anomaly'))
print("Count of 'normal':", np.sum(blackbox_classification == 'normal'))

First 10 classifications: ['normal' 'normal' 'normal' 'normal' 'normal' 'normal' 'normal' 'normal'
 'normal' 'normal']
Count of 'anomaly': 5767
Count of 'normal': 4204725


## Calculate percentages

### Subtask:
Compute the percentage of data points classified as 'anomalies' and 'normals' within the BlackBox dataset.


## Summary:

### Q&A
The anomaly threshold, set at the 99.9th percentile of `bkg_scores`, is 8.4339.

### Data Analysis Key Findings
*   The anomaly threshold was determined to be 8.4339, which corresponds to the 99.9th percentile of the background anomaly scores (`bkg_scores`).
*   Applying this threshold to the BlackBox dataset (`bb_scores`) resulted in the classification of 5,767 data points as 'anomalies'.
*   The percentage of anomalies detected in the BlackBox dataset is 0.14% of the total data points.
*   The remaining 4,204,725 data points were classified as 'normal'.

### Insights or Next Steps
*   The chosen 99.9th percentile effectively identifies a small, distinct subset of anomalies within the BlackBox dataset, which aligns with typical anomaly detection scenarios where anomalies are rare.
*   The next logical step would be to further analyze these identified anomalies to understand their characteristics or potential impact.
